In [2]:
%load_ext autoreload
%autoreload 2
from kg.cleaning.referencing import PipelineBuilder, EntityMapper
from PyPDF2 import PdfReader
import os
import amrlib
import penman
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch
from amrlib import load_stog_model
import networkx as nx
import sys, os
from pyvis.network import Network
import os
import json
import xml.etree.ElementTree as ET
import psutil
import os
import signal

import epo_ops
import spacy
from dotenv import load_dotenv
from tools.sentence.entity import Entity, InMemoryEntityRepository

from PatentProvider import PatentProvider
import random
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json
import json
import random
import os
import spacy
from spacy.tokens import DocBin
import torch
import matplotlib.pyplot as plt
import amrlib
import spacy
from tools.sentence.sentence import Sentence
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from kg.formatting.formatting_manager import FormattingManager
import sys, importlib,os
sys.modules.pop('PatentTextFormatter', None)
importlib.invalidate_caches()
import numpy as np
from kg.cleaning.normalising.word_normaliser import WordNormaliser
from fastcoref import FCoref
# Import extensions to register spaCy components
from kg.cleaning.referencing import extensions  # noqa: F401
from tools.sentence.entity import Entity
import torch, os, platform
from kg.generating_kg.generating.NodeGenerator import NodeGenerator
import spacy
import os, pyvis, jinja2, sys
import epo_ops
import epo_ops
import os
import spacy
import xml.etree.ElementTree as ET
import json
import sys
from kg.generating_kg.analysing.TextEncoder import TextEncoder
from PatentProvider import PatentProvider
# Import new graph processing classes
from tools.graph.visualizer import GraphVisualizer
from tools.graph.faiss_merger import FAISSEdgeMerger
from tools.graph.neo4j_manager import Neo4jManager
from kg.formatting.formatting_manager import FormattingManager
from tools.sentence.sentence import Sentence
from tools.sentence.sentence_classifier import SentenceClassifier
from dataclasses import dataclass
from typing import List
from kg.generating_kg.generating.ParallelTripleGenerator import ParallelTripleGenerator
from tools.graph.relation_decomposer import RelationDecomposer
import networkx as nx
import logging
import importlib
import web_editor.graph_validator_chat
#from web_editor.graph_validator_chat import start_validator_chat
from tools.graph.visualizer import GraphVisualizer
from tools.graph.kg_gen_converter import build_id_to_name_map
# Initialize decomposer
from tools.graph.relation_simplifier import RelationSimplifier
from web_editor.graph_validator_chat import start_validator_chat


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of tools.graph.claim_generator_langchain failed: Traceback (most recent call last):
  File "c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
  File "C:\Users\Caleb\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 13142: character maps to <undefined>
]


In [ ]:
# Disable auto-scroll in Jupyter output (Cursor/VS Code)
# 
# METHOD 1: Cursor/VS Code Settings (Recommended)
# 1. Press Ctrl+, (or Cmd+, on Mac) to open Settings
# 2. Search for: "jupyter output scroll" or "terminal scroll"
# 3. Look for these settings and disable them:
#    - "Terminal › Integrated: Scroll On Output" (uncheck)
#    - "Jupyter: Output Scrollback" (set to a high number or disable)
#
# METHOD 2: Manual scroll lock
# When viewing output, simply scroll up manually - this will pause auto-scroll
# Auto-scroll resumes when you scroll back to the bottom
#
# METHOD 3: Use a custom logging handler that buffers output
# (See next cell for implementation)

print("To disable auto-scroll:")
print("1. Go to Settings (Ctrl+,)")
print("2. Search 'terminal scroll' or 'jupyter output'")
print("3. Disable 'Terminal › Integrated: Scroll On Output'")
print("\nOr simply scroll up in the output panel to pause auto-scroll.")


In [2]:
import torch

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.7)  # 50% of VRAM

    torch.set_num_threads(4)
    torch.set_num_interop_threads(1)


In [ ]:
nlp = spacy.load("en_core_web_trf")   # see optimization ideas below
formatterManager = FormattingManager()
random_description = PatentProvider().getDescription("1502502")

from kg.formatting.formatting_manager import FormattingManager

# Initialize the formatting manager
# Default: 8 workers for retrieveContent, 12 for split

# Custom workers
fm = FormattingManager(num_workers=8, split_workers=12)
# Extract only invention-related sentences
sentences = fm.retrieveContent(random_description, chunk_size=1000)

# Returns a list of Sentence objects
for sentence in sentences:
    print(sentence.text)

In [ ]:

fm = FormattingManager()

# Returns List[Sentence], not List[str]
split_sentences = fm.split(sentences)




In [ ]:

# Initialize classifier (uses GPU if available, batch processing enabled)
classifier = SentenceClassifier(
    model_path="training/info/done/hf/sentence_classifier_model",
    batch_size=32,  # Adjust based on GPU memory
    use_gpu=True
)

# Filter sentences to keep only informative ones (much faster with batch processing)
sentence_split = classifier.filter_informative(
    split_sentences, 
    keep_labels=["INFORMATIVE"]  # Can also include ["INFORMATIVE", "FIGURE_RELATED"] if needed
)



In [ ]:

@dataclass(frozen=True)
class JoinedText:
    """
    Holds the concatenated text passed to spaCy
    and the starting character offset of each sentence.
    """
    text: str
    starts: List[int]


def join_sentences(sentences, sep=" "):
    """
    Join a list of Sentence objects into one string while
    tracking sentence start offsets.

    Args:
        sentences: List of Sentence objects with `.text`
        sep: Separator inserted between sentences (default: space)

    Returns:
        JoinedText(text, starts)
    """
    parts = []
    starts = []
    cur = 0

    for i, s in enumerate(sentences):
        starts.append(cur)
        parts.append(s.text)
        cur += len(s.text)

        if i < len(sentences) - 1:
            parts.append(sep)
            cur += len(sep)

    return JoinedText("".join(parts), starts)


In [ ]:
pipeline_builder = PipelineBuilder()
entity_mapper = EntityMapper(sentence_cls=Sentence)

joined = join_sentences(sentence_split, sep=" ")
doc = pipeline_builder.nlp(joined.text)

clusters = entity_mapper.map_to_sentences(doc, sentence_split, joined)

print("doc.ents:", len(doc.ents))
print("coref clusters:", len(doc._.coref_clusters))
print("entities in first sentence:", len(sentence_split[0].entities))
print(type(sentence_split[0].entities[0]))
print(sentence_split[0].entities[0])

# Collect all entities created by the mapper
all_entities: list[Entity] = []
for sentence in sentence_split:
    all_entities.extend(sentence.entities)

# Create repository and add entities
repo = InMemoryEntityRepository()
for entity in all_entities:
    repo.save(entity)

print("=== REPO CONTENTS ===")
for e in repo.getAll().values():
    print(
        f"Entity("
        f"name={e.name}, "
        f"label={e.label}, "
        f"id={e.id}, "
        f"ref={e.ref}"
        f")"
    )
for ent in doc.ents[:50]:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

# For each sentence, count overlaps
offset = 0
for i, s in enumerate(sentence_split):
    start = offset
    end = start + len(s.text)
    overlaps = [ent for ent in doc.ents if ent.start_char < end and ent.end_char > start]
    print("sentence", i, "overlaps", len(overlaps))
    offset = end + 1



In [ ]:
print(sentence_split)  

In [ ]:

# Initialize parallel triple generator
# - 10 workers for parallel processing
# - Rate limit: 900 calls/minute (stays below 1000 limit)
generator = ParallelTripleGenerator(
    repo=repo,max_workers=10,
    rate_limit_per_minute=900,
    verbose=True
)

# Generate triples from sentences (handles all parallelization, rate limiting, etc.)
triples = generator.generate(sentence_split)


In [ ]:
print(triples)

In [ ]:
# --- Merge relations per (head, tail) using FAISSEdgeMerger
# Initialize merger
merger = FAISSEdgeMerger(
    sim_threshold=0.8,
    embed_dim=256,
    ngram=3,
    keep="shortest",
)

# Merge relations
triples, merge_stats = merger.merge_relations(triples)

print("Merge stats:", merge_stats)
print("Before:", len(triples), "After:", len(triples))



In [ ]:


# Use the simplifier (Option 2 - RECOMMENDED)
simplifier = RelationSimplifier(
    max_relation_length=4,
    verbose=True
)

# Simplify triples (keeps same structure, adds properties)
triples = simplifier.simplify(triples)

In [ ]:
# --- Build + visualize a typed KG from List[Triple] using GraphVisualizer

# Initialize visualizer
visualizer = GraphVisualizer()

# Build ID -> Name map from sentence_split
id_to_name = visualizer.build_id_to_name_map(sentence_split)

# Build graph from triples
G = visualizer.build_graph(triples)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Visualize
visualizer.visualize_pyvis(G, out_file="graph_merged.html", id_to_name=id_to_name)


In [ ]:
# Save variables
%store triples
%store G
%store sentence_split

In [2]:
# 
# Or reload all stored variables at once
%store -r

In [ ]:

logging.basicConfig(
    level=logging.INFO,  # Use DEBUG for more detail
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

<module 'web_editor.graph_validator_chat.server' from 'c:\\Users\\Caleb\\Documents\\LLM Patent Claim Generator Thesis\\web_editor\\graph_validator_chat\\server.py'>

[API] GET /api/status
[127.0.0.1] "GET /api/status HTTP/1.1" 200 -
[API] GET /api/state
[127.0.0.1] "GET /api/state HTTP/1.1" 200 -
[API] GET /api/triples
[127.0.0.1] "GET /api/triples HTTP/1.1" 200 -


Killing PID 998760


In [3]:
import os
os.system("taskkill /IM node.exe /F")


0

In [3]:
import importlib
import web_editor.graph_validator_chat.server
importlib.reload(web_editor.graph_validator_chat.server)

PORT = 3000

killed_pids = set()

for conn in psutil.net_connections(kind="inet"):
    if conn.laddr and conn.laddr.port == PORT and conn.status == psutil.CONN_LISTEN:
        pid = conn.pid
        if pid and pid not in killed_pids:
            print(f"Killing PID {pid}")
            os.kill(pid, signal.SIGTERM)
            killed_pids.add(pid)
        
%store -r
# Fix Jinja2 compatibility - run this FIRST
import jinja2
if not hasattr(jinja2, ''):
    from markupsafe import escape
    jinja2.escape = escape
    import logging
import sys
import logging
# Configure logging to output to stdout (visible in Jupyter cells)
logging.basicConfig(
    level=logging.INFO,  # or logging.DEBUG for more detail
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,  # This ensures output goes to the cell
    force=True  # Override any existing c
)

# Now your logging will appear in the cell
# OP
# Now your normal imports will work
from tools.graph.kg_gen_converter import build_id_to_name_map
# Build id_to_name mapping
id_to_name = build_id_to_name_map(triples)

# Start chat (browser opens automatically)
# Use LangGraph validator (default)

# Enable debug mode
start_validator_chat(
    graph=G,
    sentence_split=sentence_split,
    triples=triples,
    id_to_name=id_to_name,
    debug=True,
    port = 50025  # <-- Enable debug mode
)

[Server] Requested API port: 50025
✓ Using requested port 50025 for API server
✓ API Server running on http://localhost:50025
✓ Using port 3000 for Next.js frontend
🚀 Starting Next.js dev server on port 3000...
✓ Graph Validator Chat: http://localhost:3000
✓ API Server: http://localhost:50025
[Server] Initializing validator in background...

[Server] ✓ Servers are running. This cell will stay ACTIVE.
[Server] ✓ Debugger will remain attached while this loop runs.
[Server] ✓ Press Ctrl+C in this cell to stop servers.
[Server] Entering blocking loop (checking every 0.5s)...
[Server] Current time: 14:42:50
[Server] About to enter loop.
[Server] _server_running = True
[Server] api thread alive = True
[Server] nextjs thread alive = True
[Server] Starting background analysis...
2026-01-15 14:42:51 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
[API] GET /api/claims/progress
[API] No active progress found. Current value: {}
[127.0.0.1] "GET /

Batches: 100%|██████████| 1/1 [00:00<00:00, 151.95it/s]


[API] GET /api/claims/progress
[API] Returning progress: {'stage': 'generating', 'message': 'Generating claim 1 (independent)...', 'progress': 20, 'current_claim': 1, 'total_claims': 9}
[127.0.0.1] "GET /api/claims/progress HTTP/1.1" 200 -


Batches: 100%|██████████| 1/1 [00:00<00:00, 280.12it/s]


✓ Built Faiss index with 181 triples


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 244.04it/s]


2026-01-15 14:47:37 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
[API] GET /api/claims/progress
[API] Returning progress: {'stage': 'refining', 'message': 'Refining claim 8 based on feedback (iteration 1/2)...', 'progress': 89}
[127.0.0.1] "GET /api/claims/progress HTTP/1.1" 200 -
[Refinement] Claim 8 refinement completed
[API] Progress: refining - Claim 8 refined (score: 60.0/100) (91%)
[Refinement] should_continue_after_refine: old_iter=1, new_iter=2, max_iterations=2
[Refinement] Reached max iterations (2) after refine, new_iter: 2, stopping
[Refinement] LangGraph step 2: refine, iteration=1
[API] Progress: refining - Refining claim 8 (iteration 1/2)... (89%)
[Refinement] LangGraph workflow completed after 2 steps
[Refinement] Claim 8 refined: score=60.0, iterations=1
[Refinement] Starting refinement for claim 9...
[Refinement] Starting LangGraph workflow for claim 9...
[Refinement] Iteration 0: Judging all claims...
[API] Progre

Batches: 100%|██████████| 1/1 [00:00<00:00, 182.50it/s]

[Server] ⏱️  Still running... (5m 5s elapsed) [Time: 14:47:55]


2026-01-15 14:47:56 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
[API] GET /api/claims/progress
[API] Returning progress: {'stage': 'refining', 'message': 'Refining claim 9 based on feedback (iteration 1/2)...', 'progress': 89}
[127.0.0.1] "GET /api/claims/progress HTTP/1.1" 200 -
[API] GET /api/claims/progress
[API] Returning progress: {'stage': 'refining', 'message': 'Refining claim 9 based on feedback (iteration 1/2)...', 'progress': 89}
[127.0.0.1] "GET /api/claims/progress HTTP/1.1" 200 -
[Refinement] Claim 9 refinement completed
[API] Progress: refining - Claim 9 refined (score: 20.0/100) (91%)
[Refinement] should_continue_after_refine: old_iter=1, new_iter=2, max_iterations=2
[Refinement] Reached max iterations (2) after refine, new_iter: 2, stopping
[Refinement] LangGraph step 2: refine, iteration=1
[API] Progress: refining - Refining claim 9 (iteration 1/2)... (89%)
[Refinement] LangGraph workflow completed after 2 steps
[R

In [ ]:
from web_editor.graph_validator_chat.helper import get_all_updates

# Get everything
updates = get_all_updates()

# Extract updated data
G_updated = updates['graph']
triples_updated = updates['triples']
entities_updated = updates['entities']
id_to_name_updated = updates['id_to_name']
changes = updates['changes']

print(f"✅ Graph: {G_updated.number_of_nodes()} nodes")
print(f"✅ Triples: {len(triples_updated)} triples")
print(f"✅ Changes: {changes}")


# Visualize the updated graph
if G_updated:
    # Option 1: Visualize the graph directly (if it's already a NetworkX graph)
    print("\n📊 Visualizing updated graph...")
    visualize_nx_browser_full(G_updated, path="updated_graph.html", id_to_name=id_to_name_updated)
elif triples_updated:
    # Option 2: Build graph from triples and visualize
    print("\n📊 Building graph from updated triples and visualizing...")
    visualizer = GraphVisualizer()
    G_from_triples = visualizer.build_graph(triples_updated, deduplicate=True)
    visualize_nx_browser_full(G_from_triples, path="updated_graph.html", id_to_name=id_to_name_updated)
else:
    print("⚠️ No graph or triples to visualize")

In [ ]:
# Reload the module to get the latest changes
import importlib
import sys

# Remove the module from cache
# Note: claim_drafting_agent, claim_extractor, claim_concept_agent, assertion_agent, and cluster_manager have been removed
if 'tools.graph.graph_rag' in sys.modules:
    del sys.modules['tools.graph.graph_rag']
if 'tools.graph' in sys.modules:
    del sys.modules['tools.graph']

# Re-import (only GraphRAG remains - other classes have been removed)
from tools.graph import GraphRAG

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("lj408/PatClaimEval-Quality", trust_remote_code=True)
model = AutoModel.from_pretrained("lj408/PatClaimEval-Quality", trust_remote_code=True).to(device)
gold_claim = "1. A computer-implemented method comprising: identifying a primary code segment; ..."
candidate_claim = "1. A computer-implemented method for managing logger source code segments in a source code development platform, ..."
res = model.score_pair(gold_claim, candidate_claim , tokenizer, device)
print(res)